In [1]:
"""
MODULE 6b — Unit Price Outlier Detection
==========================================
INPUT  : Dataset/Normalised/prices_normalised.csv  (output of Module 6a)
OUTPUT : Dataset/Normalised/outliers_flagged.csv

What this module does:
  1. Reads the normalised price file from Module 6a
  2. For each row, assigns a seasonal comparison group:
       Primary  — (Commodity x Season)  if n >= MIN_GROUP_SIZE
       Fallback — (Commodity)  global   otherwise
  3. Computes IQR fences and Z-scores per group
  4. Flags rows that breach the IQR fence
  5. Escalates severity using Z-score (MEDIUM / HIGH / CRITICAL)
  6. Tags seasonal-risk commodities
  7. Saves flagged rows to CSV and prints a ranked summary

Run Module 6a first, then this, then Module 6c for visualisations.
"""

import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict

# ──────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────
BASE_DIR      = Path.cwd()
NORMALISED_FILE = BASE_DIR / "Dataset" / "Normalised" / "prices_normalised.csv"
OUTPUT_FILE     = BASE_DIR / "Dataset" / "Normalised" / "outliers_flagged.csv"

# Minimum rows in a group before IQR stats are reliable
MIN_GROUP_SIZE = 8

# IQR multiplier — 3.0 = lenient (extreme anomalies only)
# Lower toward 1.5 to flag more
IQR_MULTIPLIER = 3.0

# Z-score thresholds
Z_CRITICAL = 4.0     # |z| >= this -> CRITICAL
Z_HIGH     = 3.0     # |z| >= this -> HIGH
# Below Z_HIGH but outside IQR fence -> MEDIUM

# Seasonal risk is detected by HS chapter prefix in Module 6a and stored in
# _seasonal_risk_flag. No keyword matching needed here — the HS chapter system
# is the authoritative classification for what counts as a seasonal commodity.


# ──────────────────────────────────────────────────────────────
# LOAD NORMALISED FILE
# ──────────────────────────────────────────────────────────────
def load_normalised(filepath: Path) -> pd.DataFrame:
    if not filepath.exists():
        raise FileNotFoundError(
            f"Normalised file not found: {filepath}\n"
            "Run Module 6a first to generate it."
        )
    df = pd.read_csv(filepath, dtype=str)
    for col in ["_unit_price", "_qty_normalised", "_conversion_factor", "_month", "_year"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    # Boolean column written by Module 6a
    if "_seasonal_risk_flag" in df.columns:
        df["_seasonal_risk_flag"] = df["_seasonal_risk_flag"].astype(str).str.lower().isin(["true","1"])
    return df


# ──────────────────────────────────────────────────────────────
# OUTLIER DETECTION
# ──────────────────────────────────────────────────────────────
def _iqr_bounds(arr: np.ndarray) -> tuple[float, float]:
    q1 = float(np.percentile(arr, 25))
    q3 = float(np.percentile(arr, 75))
    return q1 - IQR_MULTIPLIER * (q3 - q1), q3 + IQR_MULTIPLIER * (q3 - q1)


def _group_stats(arr: np.ndarray) -> dict:
    lo, hi = _iqr_bounds(arr)
    return {
        "lo":     lo,
        "hi":     hi,
        "median": float(np.median(arr)),
        "mean":   float(np.mean(arr)),
        "std":    float(np.std(arr, ddof=1)) if len(arr) > 1 else 0.0,
        "q1":     float(np.percentile(arr, 25)),
        "q3":     float(np.percentile(arr, 75)),
        "n":      len(arr),
    }


def detect_outliers(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Only work on rows with a valid unit price
    workable = df["_unit_price"].notna() & (df["_conversion_status"] == "OK")

    # Pre-build price arrays for both grouping levels
    seasonal: dict[tuple, list] = defaultdict(list)
    global_:  dict[str,   list] = defaultdict(list)

    for idx in df[workable].index:
        row   = df.loc[idx]
        price = float(row["_unit_price"])
        seasonal[(row["Commodity"], row["_season"])].append(price)
        global_[row["Commodity"]].append(price)

    s_arr = {k: np.array(v) for k, v in seasonal.items()}
    g_arr = {k: np.array(v) for k, v in global_.items()}

    # Output columns
    df["_is_outlier"]       = False
    df["_outlier_dir"]      = ""
    df["_outlier_severity"] = ""
    df["_seasonal_risk"]    = False
    df["_group_label"]      = ""
    df["_group_n"]          = 0
    df["_group_median"]     = np.nan
    df["_group_q1"]         = np.nan
    df["_group_q3"]         = np.nan
    df["_group_mean"]       = np.nan
    df["_group_std"]        = np.nan
    df["_lower_fence"]      = np.nan
    df["_upper_fence"]      = np.nan
    df["_z_score"]          = np.nan

    for idx in df[workable].index:
        row       = df.loc[idx]
        commodity = row["Commodity"]
        season    = row["_season"]
        price     = float(row["_unit_price"])

        arr_s = s_arr.get((commodity, season), np.array([]))
        arr_g = g_arr.get(commodity,           np.array([]))

        if len(arr_s) >= MIN_GROUP_SIZE:
            arr   = arr_s
            label = f"{commodity} | {season} (n={len(arr)})"
        elif len(arr_g) >= MIN_GROUP_SIZE:
            arr   = arr_g
            label = f"{commodity} | Global (n={len(arr)}, {season} had {len(arr_s)} rows)"
        else:
            df.at[idx, "_group_label"] = f"{commodity} | n={len(arr_g)} — too small"
            df.at[idx, "_group_n"]     = len(arr_g)
            continue

        st = _group_stats(arr)
        z  = (price - st["mean"]) / st["std"] if st["std"] > 0 else 0.0

        df.at[idx, "_group_label"]  = label
        df.at[idx, "_group_n"]      = st["n"]
        df.at[idx, "_group_median"] = round(st["median"], 6)
        df.at[idx, "_group_q1"]     = round(st["q1"], 6)
        df.at[idx, "_group_q3"]     = round(st["q3"], 6)
        df.at[idx, "_group_mean"]   = round(st["mean"], 6)
        df.at[idx, "_group_std"]    = round(st["std"], 6)
        df.at[idx, "_lower_fence"]  = round(st["lo"], 6)
        df.at[idx, "_upper_fence"]  = round(st["hi"], 6)
        df.at[idx, "_z_score"]      = round(z, 3)

        if price < st["lo"] or price > st["hi"]:
            df.at[idx, "_is_outlier"]  = True
            df.at[idx, "_outlier_dir"] = "HIGH" if price > st["hi"] else "LOW"

            abs_z = abs(z)
            if abs_z >= Z_CRITICAL:
                sev = "CRITICAL"
            elif abs_z >= Z_HIGH:
                sev = "HIGH"
            else:
                sev = "MEDIUM"
            df.at[idx, "_outlier_severity"] = sev

            # Seasonal risk comes from HS chapter flag computed in Module 6a.
            # True for chapters 01-16, 22, 27 (food, agri, fish, fuel).
            df.at[idx, "_seasonal_risk"] = bool(row.get("_seasonal_risk_flag", False))

    return df


# ──────────────────────────────────────────────────────────────
# PRINT RESULTS
# ──────────────────────────────────────────────────────────────
def print_results(df: pd.DataFrame) -> None:

    S1 = "=" * 90
    S2 = "-" * 90

    print("\n" + S1)
    print("  MODULE 6b - UNIT PRICE OUTLIER DETECTION")
    print(S1)

    outliers = df[df["_is_outlier"]]
    n        = len(df)
    n_out    = len(outliers)
    n_work   = (df["_unit_price"].notna() & (df["_conversion_status"] == "OK")).sum()

    n_crit = (outliers["_outlier_severity"] == "CRITICAL").sum()
    n_high = (outliers["_outlier_severity"] == "HIGH").sum()
    n_med  = (outliers["_outlier_severity"] == "MEDIUM").sum()
    n_seas = outliers["_seasonal_risk"].sum()

    # -- Summary -----------------------------------------------
    print("\n" + S2)
    print("  1. SUMMARY")
    print(S2 + "\n")

    rows_info = [
        ("Total rows in normalised file",    n,       ""),
        ("Rows with valid unit price",        n_work,  ""),
        ("",                                  None,    ""),
        ("Total outliers flagged",            n_out,   f"{n_out/n_work*100:.2f}% of valid rows" if n_work else ""),
        ("  -> CRITICAL  (|z| >= {:.0f})".format(Z_CRITICAL),
                                              n_crit,  ""),
        ("  -> HIGH      (|z| >= {:.0f})".format(Z_HIGH),
                                              n_high,  ""),
        ("  -> MEDIUM    (IQR fence only)",   n_med,   ""),
        ("  -> Seasonal Risk tag",            n_seas,  "review before discarding"),
    ]

    for label, count, note in rows_info:
        if count is None:
            print()
            continue
        note_str = f"  ({note})" if note else ""
        print(f"   {label:<45} : {count:>7,}{note_str}")

    # -- Per-commodity breakdown --------------------------------
    if n_out > 0:
        print("\n" + S2)
        print("  2. OUTLIERS BY COMMODITY  (sorted by CRITICAL count)")
        print(S2 + "\n")

        comm_summary = (
            outliers.groupby("Commodity")
            .agg(
                total        = ("_is_outlier",       "sum"),
                critical     = ("_outlier_severity", lambda x: (x == "CRITICAL").sum()),
                high         = ("_outlier_severity", lambda x: (x == "HIGH").sum()),
                seasonal     = ("_seasonal_risk",    "sum"),
                overpriced   = ("_outlier_dir",      lambda x: (x == "HIGH").sum()),
                underpriced  = ("_outlier_dir",      lambda x: (x == "LOW").sum()),
                base_unit    = ("_base_unit",        lambda x: x.mode()[0] if not x.mode().empty else "?"),
                med_price    = ("_unit_price",       "median"),
                min_price    = ("_unit_price",       "min"),
                max_price    = ("_unit_price",       "max"),
            )
            .reset_index()
            .sort_values("critical", ascending=False)
        )

        for _, r in comm_summary.iterrows():
            seas = f"  ({int(r['seasonal'])} seasonal risk)" if r["seasonal"] > 0 else ""
            bu   = r.get("base_unit", "?")
            print(f"   {r['Commodity']}  [$/{ bu}]")
            print(f"      Outliers   : {int(r['total'])}  "
                  f"(CRITICAL={int(r['critical'])}  HIGH={int(r['high'])}){seas}")
            print(f"      Direction  : overpriced={int(r['overpriced'])}  "
                  f"underpriced={int(r['underpriced'])}")
            print(f"      Price range: min={r['min_price']:.4f}  "
                  f"median={r['med_price']:.4f}  max={r['max_price']:.4f}")
            print()

    # -- Top flagged rows ---------------------------------------
    print("\n" + S2)
    print("  3. TOP FLAGGED ROWS  (up to 30, ranked by severity then |z|)")
    print(S2 + "\n")

    if outliers.empty:
        print("   No outliers detected.\n")
    else:
        ranked = (
            outliers
            .assign(_abs_z=outliers["_z_score"].abs())
            .sort_values(["_outlier_severity", "_abs_z"],
                         ascending=[True, False])
            .drop(columns=["_abs_z"])
            .head(30)
        )

        for rank, (_, r) in enumerate(ranked.iterrows(), 1):
            seas_tag = "  [SEASONAL RISK]" if r.get("_seasonal_risk") else ""
            bu       = r.get("_base_unit", "?")
            dir_str  = "OVERPRICED" if r.get("_outlier_dir") == "HIGH" else "UNDERPRICED"
            print(f"   [{rank:02d}]  {r.get('_outlier_severity')}  {dir_str}{seas_tag}")
            print(f"         Commodity   : {r.get('Commodity', '?')}")
            print(f"         Period      : {r.get('Period', '?')}  ({r.get('_season', '?')})")
            print(f"         Country     : {r.get('Country', '?')}  /  {r.get('Province', '?')}")
            print(f"         Raw         : {r.get('Quantity', '?')} [{r.get('Unit of measure', '?')}]  "
                  f"-> {r.get('_qty_normalised', '?')} {bu}")
            print(f"         Value ($)   : {r.get('Value ($)', '?')}")

            up = r.get("_unit_price")
            gm = r.get("_group_median")
            lo = r.get("_lower_fence")
            hi = r.get("_upper_fence")

            if pd.notna(up):
                print(f"         Unit price  : {float(up):.6f}  $/{bu}   "
                      f"(median={float(gm):.6f}  fence=[{float(lo):.6f} … {float(hi):.6f}])")
            print(f"         Z-score     : {r.get('_z_score', '?')}  "
                  f"(group: {r.get('_group_label', '?')})")
            print()

        if len(outliers) > 30:
            print(f"   ... and {len(outliers) - 30:,} more in the saved CSV.\n")

    # -- Season coverage table ----------------------------------
    print("\n" + S2)
    print("  4. SEASON COVERAGE  (valid rows per commodity per quarter)")
    print(f"     Groups with n < {MIN_GROUP_SIZE} fall back to global distribution.")
    print(S2 + "\n")

    valid = df[df["_unit_price"].notna() & (df["_conversion_status"] == "OK")]
    if not valid.empty and "_season" in valid.columns:
        coverage = (
            valid.groupby(["Commodity", "_season"]).size()
            .reset_index(name="n")
            .pivot(index="Commodity", columns="_season", values="n")
            .fillna(0).astype(int)
            .reset_index()
        )
        # Flag rows where any season is < MIN_GROUP_SIZE (will fall back to global)
        season_cols = [c for c in coverage.columns if c.startswith("Q")]
        coverage["fallback?"] = coverage[season_cols].apply(
            lambda row: "yes" if any(0 < v < MIN_GROUP_SIZE for v in row) else "", axis=1
        )
        print("   " + coverage.to_string(index=False))
        print()


# ──────────────────────────────────────────────────────────────
# SAVE FLAGGED FILE
# ──────────────────────────────────────────────────────────────
def save_flagged(df: pd.DataFrame) -> None:
    output_path = OUTPUT_FILE
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # Save ALL rows (including non-outliers) with detection columns
    # so Module 6c can draw complete distributions
    df.to_csv(output_path, index=False)
    n_out = df["_is_outlier"].sum()
    print(f"   Saved all rows with outlier flags: {output_path}")
    print(f"   Total rows: {len(df):,}   Outlier rows: {n_out:,}")


# ──────────────────────────────────────────────────────────────
# MAIN
# ──────────────────────────────────────────────────────────────
if __name__ == "__main__":

    print("\n" + "=" * 90)
    print("  MODULE 6b - OUTLIER DETECTION")
    print("=" * 90)

    print(f"\nLoading normalised file: {NORMALISED_FILE}")
    try:
        df = load_normalised(NORMALISED_FILE)
    except FileNotFoundError as e:
        print(f"\nERROR: {e}")
        exit(1)

    print(f"Loaded {len(df):,} rows.")

    print("Running outlier detection...")
    df = detect_outliers(df)

    print_results(df)

    print("\nSaving flagged file...")
    save_flagged(df)

    n_out  = df["_is_outlier"].sum()
    n_crit = (df["_outlier_severity"] == "CRITICAL").sum()
    print("\n" + "=" * 90)
    print("  SUMMARY:")
    print(f"    Outliers flagged : {n_out:,}  (CRITICAL={n_crit})")
    print(f"    IQR multiplier   : {IQR_MULTIPLIER}  (lenient — lower to 1.5 for more flags)")
    print(f"    Z-score critical : {Z_CRITICAL}")
    print("=" * 90)
    print("\n  DONE — run Module 6c for visualisations.\n")



  MODULE 6b - OUTLIER DETECTION

Loading normalised file: /Users/midorikawaguti/DevProject/Dashboard-BR-CA/Dataset/Normalised/prices_normalised.csv
Loaded 2,350,534 rows.
Running outlier detection...


KeyboardInterrupt: 